# Apêndice E - Fine-tuning eficiente em parâmetros com LoRA

Notebook do Apêndice E do livro *Build a Large Language Model (From
Scratch)*, de Sebastian Raschka, seguindo a ordem do notebook do autor
(`appendix-E/01_main-chapter-code/appendix-E.ipynb`).

O Apêndice E refaz o classificador de spam do Capítulo 6 com **LoRA**: em
vez de treinar o último bloco do GPT, congela o modelo inteiro e treina só
duas matrizes pequenas por camada linear. As peças vêm de `src/finetuning/`
(dataset e classificador do Cap. 6, LoRA em `lora.py`).

Pré-requisito: `data/gpt2_pretrained.pth`, gerado por
`sprints/ch06-07/experiments/carregar_pesos_pretrained.py`.

In [ ]:
import sys
from pathlib import Path

# O notebook roda a partir de Apendices/appendix-E/notebooks/: a raiz do
# projeto fica tres niveis acima.
PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import tiktoken
import torch

from src.training.dispositivo import descrever_dispositivo, escolher_dispositivo

device = escolher_dispositivo()
print("Dispositivo:", descrever_dispositivo(device))

## E.1 Introdução ao LoRA

Uma camada linear tem uma matriz de pesos **W**. O fine-tuning comum
aprende uma atualização ΔW do mesmo tamanho de W. O LoRA aproxima ΔW pelo
produto de duas matrizes estreitas:

```
W' = W + (alpha / rank) · A · B      A: (entrada × rank)   B: (rank × saída)
```

Com `rank = 16` numa camada 768 × 768 são 16 · (768 + 768) = 24.576
parâmetros em vez de 589.824. **B começa com zeros**, então no início o
modelo é exatamente o pré-treinado.

In [ ]:
from src.finetuning.lora import LinearWithLoRA, LoRALayer

torch.manual_seed(123)
linear = torch.nn.Linear(768, 768)
camada = LinearWithLoRA(linear, rank=16, alpha=16)
x = torch.rand(2, 768)

print("Parametros da Linear:", sum(p.numel() for p in linear.parameters()))
print("Parametros do LoRA  :", sum(p.numel() for p in camada.lora.parameters()))
print("Saida igual a Linear no inicio (B = 0):", torch.equal(camada(x), linear(x)))

## E.2 Preparando o dataset

Mesmo dataset e mesma divisão do Capítulo 6: SMS Spam Collection
balanceado (747 spam + 747 ham), 70% treino, 10% validação e 20% teste,
mensagens truncadas em 128 tokens.

In [ ]:
from src.finetuning.dados_cls import (
    balancear_classes,
    baixar_dataset_spam,
    carregar_spam_df,
    criar_loaders_spam,
    dividir_treino_val_teste,
)

tokenizer = tiktoken.get_encoding("gpt2")
dados = balancear_classes(carregar_spam_df(baixar_dataset_spam(PROJECT_ROOT / "data")))
treino, val, teste = dividir_treino_val_teste(dados)
train_loader, val_loader, test_loader = criar_loaders_spam(
    treino, val, teste, tokenizer, max_length=128, batch_size=8)

print(f"Treino: {len(treino)} | Validacao: {len(val)} | Teste: {len(teste)}")
print(f"Lotes: {len(train_loader)} treino, {len(val_loader)} validacao, {len(test_loader)} teste")

## E.3 Inicializando o modelo

O `GPTClassifier` do Capítulo 6 recebe os pesos do GPT-2 124M. A cabeça de
classificação (768 → 2) ainda é aleatória, então a acurácia fica perto de
50%.

In [ ]:
from src.finetuning.classificador import GPTClassifier, avaliar_classificador
from src.model.config import GPT_CONFIG_124M
from src.model.generation import generate_text_simple, text_to_token_ids, token_ids_to_text

CFG_GPT2 = dict(GPT_CONFIG_124M)
CFG_GPT2["qkv_bias"] = True

torch.manual_seed(123)
modelo = GPTClassifier(CFG_GPT2, num_classes=2)
ckpt = torch.load(PROJECT_ROOT / "data" / "gpt2_pretrained.pth", map_location="cpu")
modelo.gpt.load_state_dict(ckpt["model_state_dict"])

# O GPT dentro do classificador continua gerando texto: os pesos carregaram
ids = generate_text_simple(modelo.gpt, text_to_token_ids("Every effort moves you", tokenizer),
                           max_new_tokens=15, context_size=CFG_GPT2["context_length"])
print(token_ids_to_text(ids, tokenizer))

In [ ]:
modelo.to(device)
torch.manual_seed(123)
for nome, loader in (("treino", train_loader), ("validacao", val_loader), ("teste", test_loader)):
    print(f"Acuracia {nome:<10}: {avaliar_classificador(modelo, loader, device):.3f}")

## E.4 Fine-tuning com LoRA

`preparar_classificador_lora` congela **todos** os parâmetros (inclusive a
cabeça, como no livro) e troca cada `nn.Linear` por `LinearWithLoRA`. O
`gpt.out_head` (768 → 50.257) fica de fora: ele não é usado na
classificação e custaria 816.400 parâmetros à toa.

In [ ]:
from src.finetuning.lora import (
    contar_parametros,
    parametros_lora_esperados,
    preparar_classificador_lora,
)

_, total_original = contar_parametros(modelo)
print(f"Treinaveis antes : {sum(p.numel() for p in modelo.parameters() if p.requires_grad):,} (estrategia do cap. 6)")

preparar_classificador_lora(modelo, rank=16, alpha=16)
modelo.to(device)
treinaveis, _ = contar_parametros(modelo)
print(f"Treinaveis depois: {treinaveis:,} (livro: 2,666,528; conta a mao: {parametros_lora_esperados(CFG_GPT2, 16):,})")
print(f"Fracao treinada  : {100 * treinaveis / total_original:.2f}% dos parametros originais")
print()
print(modelo.gpt.trf_blocks[0].att.W_query)

In [ ]:
torch.manual_seed(123)
for nome, loader in (("treino", train_loader), ("validacao", val_loader), ("teste", test_loader)):
    print(f"Acuracia {nome:<10}: {avaliar_classificador(modelo, loader, device):.3f}  (igual: B = 0)")

In [ ]:
import time
from src.finetuning.classificador import treinar_classificador

torch.manual_seed(123)
optimizer = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad],
                              lr=8e-4, weight_decay=0.1)
inicio = time.time()
historico = treinar_classificador(modelo, train_loader, val_loader, optimizer, device,
                                  num_epochs=5, eval_freq=50)
print(f"Treino concluido em {(time.time() - inicio) / 60:.2f} minutos")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(historico["steps"], historico["train_losses"], label="Treino (1 lote)")
ax1.plot(historico["steps"], historico["val_losses"], linestyle="-.", label="Validacao")
ax1.set_xlabel("Passo"); ax1.set_ylabel("Perda"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(historico["steps"], historico["val_accs"])
ax2.set_xlabel("Passo"); ax2.set_ylabel("Acuracia de validacao"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
for nome, loader in (("treino", train_loader), ("validacao", val_loader), ("teste", test_loader)):
    print(f"Acuracia {nome:<10}: {avaliar_classificador(modelo, loader, device):.3f}")

### Salvar só o LoRA e mesclar nos pesos

Duas vantagens práticas do LoRA:

- **Arquivo pequeno:** basta salvar as matrizes A e B; o resto são os pesos
  do GPT-2, que já estão no checkpoint pré-treinado.
- **Sem custo na inferência:** `W + (alpha/rank)·(A·B)ᵀ` pode ser somado
  em W uma única vez, e o modelo volta a ter só camadas `nn.Linear`.

In [ ]:
from src.finetuning.lora import mesclar_lora, state_dict_lora

arquivo = PROJECT_ROOT / "models" / "appendix-E" / "lora_spam_notebook.pth"
arquivo.parent.mkdir(parents=True, exist_ok=True)
torch.save(state_dict_lora(modelo), arquivo)
mb_modelo = sum(p.numel() * p.element_size() for p in modelo.parameters()) / 1024**2
print(f"So o LoRA: {arquivo.stat().st_size / 1024**2:.1f} MB | modelo inteiro: {mb_modelo:,.0f} MB")

modelo.eval()
ids_teste, _ = next(iter(test_loader))
with torch.no_grad():
    antes = modelo(ids_teste.to(device))
    mesclar_lora(modelo)
    depois = modelo(ids_teste.to(device))
print("Camadas LoRA depois de mesclar:", sum(isinstance(m, LinearWithLoRA) for m in modelo.modules()))
print(f"Diferenca maxima nos logits: {(antes - depois).abs().max().item():.2e}")